# Ćwiczenie 6 - Sieci Konwolucyjne (CNN)

W tym ćwiczeniu wykorzystamy sieci konwolucyjne (CNN) z warstwami Conv2d i MaxPooling do klasyfikacji obrazów ze zbioru FashionMNIST.

**Cel:**
Przebadać wpływ różnych parametrów sieci CNN:
- Liczba kanałów wyjściowych warstwy konwolucyjnej
- Rozmiar filtra warstwy konwolucyjnej (kernel_size)
- Rozmiar okna poolingu
- Zaburzenia danych (szum gaussowski w danych testowych vs w testowych i treningowych)

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm

# Sprawdź czy GPU jest dostępne
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

KeyboardInterrupt: 

## Definicja modeli CNN

In [ ]:
class SimpleCNN(nn.Module):
    """
    Prosta sieć konwolucyjna z konfigurowalnymi parametrami.
    Architektura: Conv2d -> ReLU -> MaxPool -> Conv2d -> ReLU -> MaxPool -> Flatten -> Linear -> Linear
    """
    def __init__(self, num_channels=32, kernel_size=3, pool_size=2):
        super(SimpleCNN, self).__init__()
        
        # Pierwsza warstwa konwolucyjna
        # Input: (batch, 1, 28, 28) - 1 kanał (czarno-białe obrazy)
        # Output: (batch, num_channels, H', W')
        self.conv1 = nn.Conv2d(
            in_channels=1, 
            out_channels=num_channels, 
            kernel_size=kernel_size,
            padding=kernel_size//2  # padding='same'
        )
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=pool_size)
        
        # Druga warstwa konwolucyjna
        # Zwiększamy liczbę kanałów 2x
        self.conv2 = nn.Conv2d(
            in_channels=num_channels,
            out_channels=num_channels * 2,
            kernel_size=kernel_size,
            padding=kernel_size//2
        )
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=pool_size)
        
        # Spłaszczenie
        self.flatten = nn.Flatten()
        
        # Warstwy w pełni połączone
        # LazyLinear automatycznie ustala wymiarowość przy pierwszym przejściu
        self.fc1 = nn.LazyLinear(128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)  # 10 klas w FashionMNIST
    
    def forward(self, x):
        # Pierwszy blok konwolucyjny
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)
        
        # Drugi blok konwolucyjny
        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)
        
        # Spłaszczenie i warstwy liniowe
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu3(x)
        x = self.fc2(x)
        
        return x

## Funkcje pomocnicze

In [ ]:
def load_data(batch_size=64):
    """
    Ładuje zbiór FashionMNIST.
    """
    # Transformacja: konwersja do tensora i normalizacja
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))  # Normalizacja do zakresu [-1, 1]
    ])
    
    # Pobierz dane
    train_dataset = torchvision.datasets.FashionMNIST(
        root='./data',
        train=True,
        download=True,
        transform=transform
    )
    
    test_dataset = torchvision.datasets.FashionMNIST(
        root='./data',
        train=False,
        download=True,
        transform=transform
    )
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    return train_loader, test_loader


def add_gaussian_noise(batch, noise_std=0.2):
    """
    Dodaje szum gaussowski do batcha danych.
    
    Args:
        batch: Tensor z danymi (batch_size, channels, height, width)
        noise_std: Odchylenie standardowe szumu
    
    Returns:
        Batch z dodanym szumem
    """
    if noise_std > 0:
        noise = torch.randn_like(batch) * noise_std
        return batch + noise
    return batch


def train_epoch(model, train_loader, criterion, optimizer, device, train_noise_std=0.0):
    """
    Trenuje model przez jedną epokę.
    """
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    # Usunięto tqdm z pętli wewnętrznej, aby uniknąć podwójnych pasków postępu
    for inputs, labels in train_loader:
        # Dodaj szum do danych treningowych jeśli wymagany
        inputs = add_gaussian_noise(inputs, train_noise_std)
        
        inputs, labels = inputs.to(device), labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Statystyki
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = correct / total
    
    return epoch_loss, epoch_acc


def evaluate(model, test_loader, criterion, device, test_noise_std=0.0):
    """
    Ewaluuje model na zbiorze testowym.
    """
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            # Dodaj szum do danych testowych jeśli wymagany
            inputs = add_gaussian_noise(inputs, test_noise_std)
            
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / len(test_loader)
    epoch_acc = correct / total
    
    return epoch_loss, epoch_acc


def train_and_evaluate(model, train_loader, test_loader, epochs=20, lr=0.001,
                      train_noise_std=0.0, test_noise_std=0.0):
    """
    Trenuje i ewaluuje model.
    
    Returns:
        history: Słownik z historią treningu
        model: Wytrenowany model
    """
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    history = {
        'train_loss': [],
        'train_acc': [],
        'test_loss': [],
        'test_acc': []
    }
    
    # Dodano tqdm do pętli epok
    for epoch in tqdm(range(epochs), desc="Training"):
        # Trening
        train_loss, train_acc = train_epoch(
            model, train_loader, criterion, optimizer, device, train_noise_std
        )
        
        # Ewaluacja
        test_loss, test_acc = evaluate(
            model, test_loader, criterion, device, test_noise_std
        )
        
        # Zapisz historię
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_loss'].append(test_loss)
        history['test_acc'].append(test_acc)
        
        if (epoch + 1) % 5 == 0:
            # Używamy tqdm.write zamiast print, aby nie psuć paska postępu
            tqdm.write(f"Epoch {epoch+1}/{epochs} - "
                  f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} - "
                  f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")
    
    return history, model


def plot_comparison(results_list, title):
    """
    Rysuje wykresy porównawcze dla różnych konfiguracji.
    
    Args:
        results_list: Lista słowników z kluczami 'label' i 'history'
        title: Tytuł wykresów
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Wykres accuracy
    for result in results_list:
        ax1.plot(result['history']['test_acc'], label=result['label'], linewidth=2)
    
    ax1.set_xlabel('Epoch', fontsize=11)
    ax1.set_ylabel('Test Accuracy', fontsize=11)
    ax1.set_title('Test Accuracy', fontsize=12, fontweight='bold')
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.3)
    
    # Wykres loss
    for result in results_list:
        ax2.plot(result['history']['test_loss'], label=result['label'], linewidth=2)
    
    ax2.set_xlabel('Epoch', fontsize=11)
    ax2.set_ylabel('Test Loss', fontsize=11)
    ax2.set_title('Test Loss', fontsize=12, fontweight='bold')
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)
    
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


def print_summary_table(results_list):
    """
    Wyświetla tabelę podsumowującą wyniki.
    """
    print("\n" + "="*90)
    print("PODSUMOWANIE WYNIKÓW")
    print("="*90)
    print(f"{'Konfiguracja':<45} {'Train Acc':>12} {'Test Acc':>12} {'Gap':>12}")
    print("-"*90)
    
    for result in results_list:
        train_acc = result['history']['train_acc'][-1]
        test_acc = result['history']['test_acc'][-1]
        gap = train_acc - test_acc
        print(f"{result['label']:<45} {train_acc:>12.4f} {test_acc:>12.4f} {gap:>12.4f}")
    
    print("="*90)

## Załaduj dane

Używamy standardowego batch_size=64 dla wszystkich eksperymentów.

In [ ]:
# Załaduj dane z domyślnym batch_size
train_loader, test_loader = load_data(batch_size=64)

print(f"Liczba batchy treningowych: {len(train_loader)}")
print(f"Liczba batchy testowych: {len(test_loader)}")
print(f"Rozmiar datasetu treningowego: {len(train_loader.dataset)}")
print(f"Rozmiar datasetu testowego: {len(test_loader.dataset)}")

## Eksperyment 1: Liczba kanałów wyjściowych

Przebadamy wpływ liczby kanałów w pierwszej warstwie konwolucyjnej:
- 16 kanałów (mała sieć)
- 32 kanały (średnia sieć)
- 64 kanały (duża sieć)

Druga warstwa zawsze ma 2x więcej kanałów niż pierwsza.

In [ ]:
print("\n=== Eksperyment 1: Liczba kanałów ===")

results_channels = []
channel_configs = [16, 32, 64]

for num_channels in channel_configs:
    print(f"\n--- Trenowanie modelu z {num_channels} kanałami ---")
    
    model = SimpleCNN(num_channels=num_channels, kernel_size=3, pool_size=2)
    history, trained_model = train_and_evaluate(
        model, train_loader, test_loader, epochs=20, lr=0.001
    )
    
    results_channels.append({
        'label': f'{num_channels} channels',
        'history': history
    })

plot_comparison(results_channels, "Eksperyment 1: Wpływ liczby kanałów")
print_summary_table(results_channels)

### Wnioski - Eksperyment 1:

**Do uzupełnienia po uruchomieniu:**
- Jak liczba kanałów wpływa na accuracy?
- Czy większa liczba kanałów = lepsze wyniki?
- Czy jest ryzyko overfittingu przy dużej liczbie kanałów?
- Jaki jest trade-off między wydajnością a dokładnością?

## Eksperyment 2: Rozmiar filtra (kernel_size)

Przebadamy wpływ rozmiaru filtra konwolucyjnego:
- kernel_size = 3 (3x3 filtr - standard)
- kernel_size = 5 (5x5 filtr - większe pole widzenia)
- kernel_size = 7 (7x7 filtr - bardzo duże pole widzenia)

In [ ]:
print("\n=== Eksperyment 2: Rozmiar filtra ===")

results_kernel = []
kernel_configs = [1, 3, 5, 7, 9]

for kernel_size in kernel_configs:
    print(f"\n--- Trenowanie modelu z kernel_size={kernel_size} ---")
    
    model = SimpleCNN(num_channels=32, kernel_size=kernel_size, pool_size=2)
    history, trained_model = train_and_evaluate(
        model, train_loader, test_loader, epochs=20, lr=0.001
    )
    
    results_kernel.append({
        'label': f'kernel_size={kernel_size}',
        'history': history
    })

plot_comparison(results_kernel, "Eksperyment 2: Wpływ rozmiaru filtra")
print_summary_table(results_kernel)

### Wnioski - Eksperyment 2:

**Do uzupełnienia po uruchomieniu:**
- Czy większe filtry = lepsze wyniki?
- Jak rozmiar filtra wpływa na pole recepcyjne?
- Czy małe filtry (3x3) wystarczają dla FashionMNIST?
- Jaki jest trade-off między rozmiarem filtra a liczbą parametrów?

## Eksperyment 3: Rozmiar okna poolingu

Przebadamy wpływ rozmiaru okna MaxPooling:
- pool_size = 2 (2x2 pooling - standard, 4x redukcja)
- pool_size = 3 (3x3 pooling - bardziej agresywna redukcja)
- pool_size = 4 (4x4 pooling - bardzo agresywna redukcja)

In [ ]:
print("\n=== Eksperyment 3: Rozmiar poolingu ===")

results_pooling = []
pool_configs = [1, 2, 3, 4, 8]

for pool_size in pool_configs:
    print(f"\n--- Trenowanie modelu z pool_size={pool_size} ---")
    
    model = SimpleCNN(num_channels=32, kernel_size=3, pool_size=pool_size)
    history, trained_model = train_and_evaluate(
        model, train_loader, test_loader, epochs=20, lr=0.001
    )
    
    results_pooling.append({
        'label': f'pool_size={pool_size}',
        'history': history
    })

plot_comparison(results_pooling, "Eksperyment 3: Wpływ rozmiaru poolingu")
print_summary_table(results_pooling)

### Wnioski - Eksperyment 3:

**Do uzupełnienia po uruchomieniu:**
- Jak rozmiar poolingu wpływa na accuracy?
- Czy większy pooling = szybsze uczenie?
- Czy tracimy ważne informacje przy dużym poolingu?
- Jaki pooling jest optymalny dla obrazków 28x28?

## Eksperyment 4: Zaburzenia danych (szum gaussowski)

Przebadamy różne scenariusze dodawania szumu:
1. **Bez szumu** - baseline
2. **Szum tylko w danych testowych** - sprawdza odporność modelu
3. **Szum w train i test** - data augmentation
4. **Szum tylko w train** - regularyzacja poprzez augmentację

Używamy umiarkowanego szumu (std=0.2) i silnego szumu (std=0.5).

In [ ]:
print("\n=== Eksperyment 4: Szum w danych ===")

results_noise = []

noise_scenarios = [
    (0.0, 0.0, 'No noise (baseline)'),
    (0.0, 0.2, 'Noise only in test (std=0.2)'),
    (0.2, 0.2, 'Noise in train & test (std=0.2)'),
    (0.2, 0.0, 'Noise only in train (std=0.2)'),
    (0.0, 0.5, 'High noise only in test (std=0.5)'),
    (0.5, 0.5, 'High noise in train & test (std=0.5)'),
    (0.5, 0.0, 'High noise only in train (std=0.5)')
]

for train_noise, test_noise, label in noise_scenarios:
    print(f"\n--- {label} ---")
    
    model = SimpleCNN(num_channels=32, kernel_size=3, pool_size=2)
    history, trained_model = train_and_evaluate(
        model, train_loader, test_loader, epochs=20, lr=0.001,
        train_noise_std=train_noise, test_noise_std=test_noise
    )
    
    results_noise.append({
        'label': label,
        'history': history
    })

plot_comparison(results_noise, "Eksperyment 4: Wpływ szumu gaussowskiego")
print_summary_table(results_noise)

### Wnioski - Eksperyment 4:

**Do uzupełnienia po uruchomieniu:**
- Czy CNN są bardziej odporne na szum niż zwykłe sieci?
- Czy trening z szumem poprawia generalizację?
- Jak silny szum wpływa na uczenie?
- Która strategia (train/test/both) jest najlepsza?
- Czy szum w train działa jako regularizator?

## Eksperyment 5: Porównanie CNN vs Fully Connected

Bonus: Porównajmy sieć CNN z prostą siecią w pełni połączoną (jak w ćwiczeniu 5).

In [ ]:
class SimpleFC(nn.Module):
    """
    Prosta sieć fully connected do porównania z CNN.
    """
    def __init__(self):
        super(SimpleFC, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28*28, 128)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(128, 64)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(64, 10)
    
    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)
        return x


print("\n=== Eksperyment 5: CNN vs Fully Connected ===")

results_comparison = []

# Trenuj CNN
print("\n--- Trenowanie CNN ---")
cnn_model = SimpleCNN(num_channels=32, kernel_size=3, pool_size=2)
history_cnn, _ = train_and_evaluate(
    cnn_model, train_loader, test_loader, epochs=15, lr=0.001
)
results_comparison.append({
    'label': 'CNN (32 channels)',
    'history': history_cnn
})

# Trenuj FC
print("\n--- Trenowanie Fully Connected ---")
fc_model = SimpleFC()
history_fc, _ = train_and_evaluate(
    fc_model, train_loader, test_loader, epochs=15, lr=0.001
)
results_comparison.append({
    'label': 'Fully Connected (128->64)',
    'history': history_fc
})

plot_comparison(results_comparison, "Eksperyment 5: CNN vs Fully Connected")
print_summary_table(results_comparison)

# Policz liczbę parametrów
cnn_params = sum(p.numel() for p in cnn_model.parameters())
fc_params = sum(p.numel() for p in fc_model.parameters())

print(f"\nLiczba parametrów:")
print(f"CNN: {cnn_params:,}")
print(f"Fully Connected: {fc_params:,}")

### Wnioski - Eksperyment 5:

**Do uzupełnienia po uruchomieniu:**
- Która architektura jest lepsza?
- Jak liczba parametrów wpływa na wyniki?
- Czy CNN szybciej się uczą?
- Jakie są zalety CNN dla danych obrazowych?

## Podsumowanie końcowe

### Najważniejsze wnioski:

**1. Liczba kanałów:**
- [do uzupełnienia]

**2. Rozmiar filtra:**
- [do uzupełnienia]

**3. Rozmiar poolingu:**
- [do uzupełnienia]

**4. Szum w danych:**
- [do uzupełnienia]

**5. CNN vs FC:**
- [do uzupełnienia]

### Optymalna konfiguracja:
- Liczba kanałów: [do uzupełnienia]
- Kernel size: [do uzupełnienia]
- Pool size: [do uzupełnienia]
- Strategia szumu: [do uzupełnienia]